# 04 Ein Tool ergänzen und den Assistenten prüfen

LangChain durch Literate Programming lernen

## Konzept vor dem Code

    Eine Dokumentfrage hat einen festen Ablauf: suchen, Kontext aufbauen, antworten.
    Dafür reicht die Chain aus Notebook 03. Für eine Rechenfrage kann dagegen ein Werkzeug sinnvoll
    sein. Ich ergänze einen Lernzeitrechner, der ECTS mit einem ausdrücklich gewählten Planungswert
    von 30 Stunden multipliziert. Dieser Wert ist eine Projektannahme, keine Aussage aus den PDFs.

    Ein ReAct-Agent wählt ein Werkzeug, erhält dessen Ergebnis und entscheidet über den nächsten
    Schritt: `Entscheidung → Action → Observation → nächste Entscheidung oder Antwort`.
    Der klassische Prompt nennt die kurze Entscheidung «Thought». Das ist eine ausgegebene
    Arbeitsnotiz zur Toolwahl, kein Einblick in interne Modellüberlegungen.
    Ich protokolliere Toolname, Eingabe und Ergebnis. Ein Iterationslimit verhindert endlose Schleifen.

    Die Aufgabe nennt AgentExecutor. Deshalb verwende ich hier die kompatible Schnittstelle aus
    langchain-classic. Aktuelle LangChain-Projekte können stattdessen create_agent verwenden.
    Diese bewusste Auswahl hält den Bezug zur Aufgabenstellung sichtbar.
    Dokumentation: [Tools](https://docs.langchain.com/oss/python/langchain/tools),
    [aktuelle Agenten](https://docs.langchain.com/oss/python/langchain/overview),
    [Classic API](https://reference.langchain.com/python/langchain-classic/agents/).

## Lernziele

Nach diesem Notebook kann ich:

- ein eigenes Tool mit Eingabevalidierung definieren
- den ReAct Ablauf anhand protokollierter Toolschritte erklären
- eine RAG Chain mit einem begrenzten AgentExecutor verbinden
- Testfragen und Bewertungsregeln getrennt von Ergebnissen festlegen
- den Zusatzaufwand eines Agenten gegenüber einer festen Chain begründen

## Setup

Die folgende Zelle lädt die Einstellungen und zeigt die verwendeten Versionen. Das Modell wird erst in den Experimenten aufgerufen. So lässt sich ein Installationsproblem von einem Problem mit einer Modellantwort unterscheiden. Der Begriff hinter dem Experiment ist bereits oben eingeführt. Alle Notebooks laufen unabhängig in einem frischen Kernel.

In [1]:
import sys, os, json
from pathlib import Path
from importlib.metadata import version
from dotenv import load_dotenv
from modul_assistent.core import ROOT, mode, chat_model

load_dotenv(ROOT / '.env', override=True)
assert (3, 12) <= sys.version_info[:2] < (3, 13), 'Getestete Umgebung: Python 3.12'
for package in ['langchain-core', 'langchain-openai', 'langchain-classic']:
    print(package, version(package))
print('Betriebsart:', mode())

langchain-core 1.7.3
langchain-openai 1.7.3
langchain-classic 1.0.8
Betriebsart: demo


## Experiment

Dieses Notebook baut seinen Suchindex selbst auf. Es braucht keine Variablen aus Notebook 03. Das Tool lernzeit wird mit @tool definiert; seine Beschreibung erklärt dem Agenten, wann er es verwenden soll.

In [2]:
from modul_assistent.core import build_index, ask_rag, make_agent
from langchain_core.tools import tool

@tool
def lernzeit(ects: float) -> str:
    """Berechnet einen PLANUNGSWERT von 30 Stunden pro ECTS, keine offizielle Modulvorgabe."""
    ects = float(ects)  # Textbasiertes ReAct kann die Eingabe als Zeichenkette liefern.
    if not 0 < ects <= 30:
        raise ValueError('ECTS muss grösser als 0 und höchstens 30 sein')
    return f'Planungswert: {ects*30:g} Arbeitsstunden bei 30 Stunden pro ECTS.'

store, chunks = build_index()
print(lernzeit.name)
print(lernzeit.description)
print(lernzeit.invoke({'ects':3}))

lernzeit
Berechnet einen PLANUNGSWERT von 30 Stunden pro ECTS, keine offizielle Modulvorgabe.
Planungswert: 90 Arbeitsstunden bei 30 Stunden pro ECTS.


Der Agent erhält die Dokumentensuche und den Lernzeitrechner als Werkzeuge. Im Demo-Modus wird genau ein festgelegter Rechenablauf abgespielt. Dadurch kann ich die Verbindung zwischen AgentExecutor und Tool prüfen; autonome Modellentscheidungen belegt dieser Ablauf nicht.

In [3]:
agent = make_agent(store, calculation_tool=lernzeit)
lauf = agent.invoke({'input':'Berechne einen Planungswert für 3 ECTS mit dem Tool lernzeit.'})
print(lauf['output'])
for action, observation in lauf['intermediate_steps']:
    print('Entscheidung:',action.log)
    print('Action:',action.tool,'Eingabe:',action.tool_input)
    print('Observation:',observation)
assert any(action.tool == 'lernzeit' for action,_ in lauf['intermediate_steps'])

Planungswert: 90 Arbeitsstunden bei 30 Stunden pro ECTS.
Entscheidung: Thought: Ich berechne einen Planungswert.
Action: lernzeit
Action Input: 3
Action: lernzeit Eingabe: 3
Observation: Planungswert: 90 Arbeitsstunden bei 30 Stunden pro ECTS.


Im BFH-Modus lasse ich den Agenten zusätzlich eine Dokumentfrage bearbeiten. Im Demonstrationsmodus wird dieser echte Modellversuch übersprungen und nicht durch eine passende Erfolgsmeldung ersetzt.

In [4]:
if mode() == 'bfh':
    dokument_lauf = agent.invoke({'input':'Wie viele Notebooks sind gefordert? Suche in den Unterlagen und nenne die Quelle.'})
    print(dokument_lauf['output'])
    for action,observation in dokument_lauf['intermediate_steps']:
        print(action.tool,action.tool_input,observation)
else:
    print('Echte Toolwahl und Dokumentantwort: erst im BFH-Modus messbar.')

Echte Toolwahl und Dokumentantwort: erst im BFH-Modus messbar.


### Der Mini Assistent mit Gesprächsverlauf

Die Sitzungsverwaltung aus Notebook 02 kommt jetzt um den Agenten aus diesem Notebook.
Damit enthält der Mini-Assistent Verlauf, PDF-Suche und ein Rechentool. Der Wrapper speichert
Frage und endgültige Antwort; technische Zwischenschritte werden nicht als Gespräch ausgegeben.
Die RAG-Suche selbst bekommt die Frage, die der Agent dem Tool übergibt. Mehrdeutige Folgefragen
brauchen deshalb eine passende Umformulierung durch den Agenten. Das ist im BFH-Modus zu prüfen.
Die folgende lokale Probe belegt die Speicherung einer Sitzung, keine gelungene Modellinterpretation.

In [5]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
gespraeche = {}
def get_history(session_id):
    return gespraeche.setdefault(session_id, InMemoryChatMessageHistory())
mini_assistent = RunnableWithMessageHistory(agent, get_history,
    input_messages_key='input', history_messages_key='history', output_messages_key='output')
def fragen(frage, session_id='meine-sitzung'):
    return mini_assistent.invoke({'input':frage},
        config={'configurable':{'session_id':session_id}})
probe = fragen('Berechne den Planungswert für 3 ECTS.')
print(probe['output'])
assert len(get_history('meine-sitzung').messages) == 2
assert len(get_history('andere-sitzung').messages) == 0

Planungswert: 90 Arbeitsstunden bei 30 Stunden pro ECTS.


C:\Users\linas\Documents\Codex\2026-10-30\applications-mentioned-by-the-user-appshot\work\venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)
C:\Users\linas\Documents\Codex\2026-10-30\applications-mentioned-by-the-user-appshot\work\ipykernel_22720\2923809810.py:5: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  return gespraeche.setdefault(session_id, InMemoryChatMessageHistory())


## Mini Evaluation

Vor den Antworten lege ich die Fälle fest. Dokumentfragen bekommen eine Sollseite, die Rechenfrage einen Sollwert, und eine unbeantwortbare Frage soll Nichtwissen auslösen. Die Prüfungen unten erfassen nur einfache Kriterien. Gerade die inhaltliche Belegtreue muss ich zusätzlich selbst lesen.

In [6]:
testfaelle = [
 {'id':'Q1','frage':'Wie viele eigenständig lauffähige Notebooks sind gefordert?', 'typ':'fakt','seite':1,'signal':'vier'},
 {'id':'Q2','frage':'Wie lang soll die Abschlussreflexion sein?', 'typ':'fakt','seite':4,'signal':'1'},
 {'id':'Q3','frage':'Wie viele typische Fehler sollen pro Notebook gezeigt werden?', 'typ':'fakt','seite':2,'signal':'Fehler'},
 {'id':'Q4','frage':'In welcher Sprache muss die Abschlusspräsentation gehalten werden?', 'typ':'unbekannt','seite':None,'signal':'nicht'},
 {'id':'Q5','frage':'Berechne den Planungswert für 3 ECTS.', 'typ':'tool','seite':None,'signal':'90'},
]
print(json.dumps(testfaelle,ensure_ascii=False,indent=2))

[
  {
    "id": "Q1",
    "frage": "Wie viele eigenständig lauffähige Notebooks sind gefordert?",
    "typ": "fakt",
    "seite": 1,
    "signal": "vier"
  },
  {
    "id": "Q2",
    "frage": "Wie lang soll die Abschlussreflexion sein?",
    "typ": "fakt",
    "seite": 4,
    "signal": "1"
  },
  {
    "id": "Q3",
    "frage": "Wie viele typische Fehler sollen pro Notebook gezeigt werden?",
    "typ": "fakt",
    "seite": 2,
    "signal": "Fehler"
  },
  {
    "id": "Q4",
    "frage": "In welcher Sprache muss die Abschlusspräsentation gehalten werden?",
    "typ": "unbekannt",
    "seite": null,
    "signal": "nicht"
  },
  {
    "id": "Q5",
    "frage": "Berechne den Planungswert für 3 ECTS.",
    "typ": "tool",
    "seite": null,
    "signal": "90"
  }
]


Die RAG-Fälle prüfen Antwortform, Sollsignal und Seitenangabe. Ich speichere Modus und Resultat gemeinsam, damit Demo-Prüfungen nicht als BFH-Messung erscheinen. Die Agentenwahl wird für den Rechenfall separat aus den tatsächlichen Zwischenschritten bewertet. Laufzeiten werden gemessen, nicht geschätzt.

In [7]:
import time, csv
resultate = []
for fall in testfaelle:
    startzeit = time.perf_counter()
    if fall['typ'] == 'tool':
        ausgabe = agent.invoke({'input':fall['frage']})
        text = ausgabe['output']
        quelle = 'Planungsannahme 30 Stunden pro ECTS'
        tool_ok = any(a.tool == 'lernzeit' for a,_ in ausgabe['intermediate_steps'])
    else:
        antwort, dokumente = ask_rag(fall['frage'],store,k=8)
        text,quelle = antwort.antwort,antwort.quelle
        tool_ok = None
    signal_ok = fall['signal'].lower() in text.lower()
    seite_ok = (f"S. {fall['seite']}" in quelle) if fall['seite'] else None
    unbekannt_ok = (quelle == 'keine Quelle' and signal_ok) if fall['typ']=='unbekannt' else None
    resultate.append({'id':fall['id'],'modus':mode(),'antwort':text,'quelle':quelle,
        'signal_ok':signal_ok,'sollseite_ok':seite_ok,'tool_ok':tool_ok,
        'nichtwissen_ok':unbekannt_ok,'sekunden':round(time.perf_counter()-startzeit,3),
        'inhalt_manuell':'noch prüfen'})
ziel = ROOT/'ergebnisse'/f'{"live" if mode()=="bfh" else "demo"}_evaluation.csv'
with ziel.open('w',encoding='utf-8-sig',newline='') as datei:
    writer = csv.DictWriter(datei,fieldnames=list(resultate[0])); writer.writeheader(); writer.writerows(resultate)
for zeile in resultate: print(zeile)
print('Gespeichert:',ziel.name)

{'id': 'Q1', 'modus': 'demo', 'antwort': 'Gefordert sind vier eigenständig lauffähige Jupyter Notebooks.', 'quelle': 'LangChain_Literate_Programming_Aufgabenstellung.pdf, S. 1', 'signal_ok': True, 'sollseite_ok': True, 'tool_ok': None, 'nichtwissen_ok': None, 'sekunden': 0.016, 'inhalt_manuell': 'noch prüfen'}
{'id': 'Q2', 'modus': 'demo', 'antwort': 'Die Abschlussreflexion soll ungefähr 1 bis 2 Seiten umfassen.', 'quelle': 'LangChain_Literate_Programming_Aufgabenstellung.pdf, S. 4', 'signal_ok': True, 'sollseite_ok': True, 'tool_ok': None, 'nichtwissen_ok': None, 'sekunden': 0.008, 'inhalt_manuell': 'noch prüfen'}
{'id': 'Q3', 'modus': 'demo', 'antwort': 'Pro Notebook soll mindestens ein typischer Fehler reproduziert, erklärt und korrigiert werden.', 'quelle': 'LangChain_Literate_Programming_Aufgabenstellung.pdf, S. 2', 'signal_ok': True, 'sollseite_ok': True, 'tool_ok': None, 'nichtwissen_ok': None, 'sekunden': 0.006, 'inhalt_manuell': 'noch prüfen'}
{'id': 'Q4', 'modus': 'demo', 'an

Die Tabelle ist ein Prüfprotokoll, keine umfassende Genauigkeitsmessung. Fünf bekannte Fälle erlauben keine Aussage über alle Modulfragen. Für einen Modellvergleich würde ich jede Frage mehrfach mit gleicher Konfiguration prüfen und zusätzlich Antworttreue, Quellenbezug und angemessenes Nichtwissen manuell bewerten.

In [8]:
for zeile in resultate:
    print(zeile['id'], 'Signal:',zeile['signal_ok'],
          'Seite:',zeile['sollseite_ok'],'Tool:',zeile['tool_ok'],
          'Nichtwissen:',zeile['nichtwissen_ok'])
if mode() == 'demo':
    assert all(z['signal_ok'] for z in resultate)
    assert all(z['sollseite_ok'] for z in resultate if z['sollseite_ok'] is not None)
    assert resultate[3]['nichtwissen_ok'] and resultate[4]['tool_ok']

Q1 Signal: True Seite: True Tool: None Nichtwissen: None
Q2 Signal: True Seite: True Tool: None Nichtwissen: None
Q3 Signal: True Seite: True Tool: None Nichtwissen: None
Q4 Signal: True Seite: None Tool: None Nichtwissen: True
Q5 Signal: True Seite: None Tool: True Nichtwissen: None


## Häufige Fehler

Ein Tool darf ungültige Werte nicht stillschweigend verarbeiten. Ich übergebe absichtlich negative ECTS. Der Fehler entsteht im Tool und lässt sich unabhängig vom Agenten untersuchen. Die Korrektur ist eine zulässige Zahl.

In [9]:
try:
    lernzeit.invoke({'ects':-3})
except ValueError as fehler:
    print(type(fehler).__name__,str(fehler))
else:
    raise AssertionError('Negative ECTS wurden akzeptiert')
assert '90' in lernzeit.invoke({'ects':3})

ValueError ECTS muss grösser als 0 und höchstens 30 sein


## Prompt Logbuch

**Mein eigener erster Prompt ohne KI-Hilfe:**

> Noch von mir einzutragen. Die folgende Endversion ist ein KI-gestützter Vorschlag. Ein nachträglich erfundener Erstprompt würde die geforderte eigene Iteration nicht belegen.

**Vorgeschlagene Endversion:**

```text
Beantworte Dokumentfragen mit modul_suche und gib die dort genannte Quelle unverändert weiter. Verwende für die Lernzeit lernzeit und kennzeichne das Ergebnis als Planungswert. Erfinde keine Modulvorgaben. Beende den Ablauf, sobald die Frage beantwortet ist.
```

**Begründung des Vorschlags:** Toolwahl und Wissensquelle müssen getrennt beschrieben sein. Ein Rechner liefert einen exakten Wert unter einer Annahme; dadurch wird die Annahme nicht zur offiziellen Regel.

**Mein Vergleich nach dem eigenen Versuch:** Hier trage ich Modell, Datum, beide tatsächlichen Antworten und meine Beobachtung ein. Demo-Ausgaben sind kein Modellversuch.

## Zusammenfassung

Der Mini-Assistent verbindet RAG und ein Rechentool. Für reine Dokumentfragen bleibt die feste Chain übersichtlicher. Der Agent lohnt sich erst, wenn die passende Aktion von der Frage abhängt. Die lokalen Tests prüfen bekannte Abläufe; die Qualität des BFH-Modells muss mit echten Durchläufen ergänzt werden. Die Abschlussreflexion und zwei Erweiterungen stehen in docs/Abschlussreflexion.md.

**Selbstkontrolle:** Kann ich die oben genannten Lernziele anhand der ausgeführten Zellen erklären, ohne die Hilfsfunktionen nur abzulesen?

## Übungsaufgabe

Formuliere zwei neue beantwortbare Dokumentfragen und eine weitere unbeantwortbare Frage. Lege Sollseite und Bewertung vor dem Aufruf fest. Ergänze im BFH-Modus die tatsächlichen Antworten und lies jede genannte PDF-Seite nach. Lege für den Demo-Modus neue Fälle ausdrücklich als selbst geschriebene Testdaten an.

Die folgende Zelle bleibt absichtlich leer und ist für meine eigene Lösung vorgesehen.